# 03 — Gold Feature Engineering

Reads from Silver Delta table, engineers all model features (urgency scores, school holidays),
and writes to Gold. Includes EDA. Depends on `02_silver` completing successfully.

In [ ]:
%pip install pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (16, 5)
sns.set_theme(style='whitegrid')

In [ ]:
CATALOG = 'workspace'
SCHEMA  = 'default'

In [ ]:
# --- Segments: display name -> Google Trends keyword ---
SEGMENTS = {
    'Primary Math':        'primary math tuition singapore',
    'Primary English':     'primary english tuition singapore',
    'Primary Science':     'primary science tuition singapore',
    'Secondary Math':      'secondary math tuition singapore',
    'Secondary English':   'secondary english tuition singapore',
    'Secondary Biology':   'secondary biology tuition singapore',
    'Secondary Chemistry': 'secondary chemistry tuition singapore',
    'Secondary Physics':   'secondary physics tuition singapore',
    'JC Math':             'jc math tuition singapore',
    'JC Chemistry':        'jc chemistry tuition singapore',
    'JC Economics':        'jc economics tuition singapore',
}

SEG_ORDER = [
    'Primary Math','Primary English','Primary Science',
    'Secondary Math','Secondary English','Secondary Biology','Secondary Chemistry','Secondary Physics',
    'JC Math','JC Chemistry','JC Economics',
]

# --- MOE school term and holiday dates 2020-2026 ---
# Source: https://www.moe.gov.sg/news/press-releases
MOE_CALENDAR = {
    2020: {
        'term1': ('2020-01-02','2020-03-13'), 'term2': ('2020-03-23','2020-05-29'),
        'term3': ('2020-06-29','2020-09-04'), 'term4': ('2020-09-14','2020-11-20'),
        'march_hols': ('2020-03-14','2020-03-22'), 'june_hols': ('2020-05-30','2020-06-28'),
        'sept_hols': ('2020-09-05','2020-09-13'),  'year_end':  ('2020-11-21','2020-12-31'),
    },
    2021: {
        'term1': ('2021-01-04','2021-03-12'), 'term2': ('2021-03-22','2021-05-28'),
        'term3': ('2021-06-28','2021-09-03'), 'term4': ('2021-09-13','2021-11-19'),
        'march_hols': ('2021-03-13','2021-03-21'), 'june_hols': ('2021-05-29','2021-06-27'),
        'sept_hols': ('2021-09-04','2021-09-12'),  'year_end':  ('2021-11-20','2021-12-31'),
    },
    2022: {
        'term1': ('2022-01-03','2022-03-11'), 'term2': ('2022-03-21','2022-05-27'),
        'term3': ('2022-06-27','2022-09-02'), 'term4': ('2022-09-12','2022-11-18'),
        'march_hols': ('2022-03-12','2022-03-20'), 'june_hols': ('2022-05-28','2022-06-26'),
        'sept_hols': ('2022-09-03','2022-09-11'),  'year_end':  ('2022-11-19','2022-12-31'),
    },
    2023: {
        'term1': ('2023-01-03','2023-03-10'), 'term2': ('2023-03-20','2023-05-26'),
        'term3': ('2023-06-26','2023-09-01'), 'term4': ('2023-09-11','2023-11-17'),
        'march_hols': ('2023-03-11','2023-03-19'), 'june_hols': ('2023-05-27','2023-06-25'),
        'sept_hols': ('2023-09-02','2023-09-10'),  'year_end':  ('2023-11-18','2023-12-31'),
    },
    2024: {
        'term1': ('2024-01-02','2024-03-08'), 'term2': ('2024-03-18','2024-05-24'),
        'term3': ('2024-06-24','2024-08-30'), 'term4': ('2024-09-09','2024-11-15'),
        'march_hols': ('2024-03-09','2024-03-17'), 'june_hols': ('2024-05-25','2024-06-23'),
        'sept_hols': ('2024-08-31','2024-09-08'),  'year_end':  ('2024-11-16','2024-12-31'),
    },
    2025: {
        'term1': ('2025-01-02','2025-03-14'), 'term2': ('2025-03-24','2025-05-30'),
        'term3': ('2025-06-30','2025-09-05'), 'term4': ('2025-09-15','2025-11-21'),
        'march_hols': ('2025-03-15','2025-03-23'), 'june_hols': ('2025-05-31','2025-06-29'),
        'sept_hols': ('2025-09-06','2025-09-14'),  'year_end':  ('2025-11-22','2025-12-31'),
    },
    2026: {
        'term1': ('2026-01-02','2026-03-13'), 'term2': ('2026-03-23','2026-05-29'),
        'term3': ('2026-06-29','2026-09-04'), 'term4': ('2026-09-14','2026-11-20'),
        'march_hols': ('2026-03-14','2026-03-22'), 'june_hols': ('2026-05-30','2026-06-28'),
        'sept_hols': ('2026-09-05','2026-09-13'),  'year_end':  ('2026-11-21','2026-12-31'),
    },
}

# --- SEAB exam and results dates ---
# Source: https://www.seab.gov.sg/important-dates-for-candidates
SEAB_EVENTS = [
    {'event':'SA1_exam',       'start':'2020-04-27','end':'2020-05-08'},
    {'event':'SA1_exam',       'start':'2021-04-26','end':'2021-05-07'},
    {'event':'SA1_exam',       'start':'2022-04-25','end':'2022-05-06'},
    {'event':'SA1_exam',       'start':'2023-04-24','end':'2023-05-05'},
    {'event':'SA1_exam',       'start':'2024-04-22','end':'2024-05-03'},
    {'event':'SA1_exam',       'start':'2025-04-28','end':'2025-05-09'},
    {'event':'SA1_exam',       'start':'2026-04-27','end':'2026-05-08'},
    {'event':'SA2_exam',       'start':'2020-09-28','end':'2020-10-16'},
    {'event':'SA2_exam',       'start':'2021-09-27','end':'2021-10-15'},
    {'event':'SA2_exam',       'start':'2022-09-26','end':'2022-10-14'},
    {'event':'SA2_exam',       'start':'2023-09-25','end':'2023-10-13'},
    {'event':'SA2_exam',       'start':'2024-09-23','end':'2024-10-11'},
    {'event':'SA2_exam',       'start':'2025-09-29','end':'2025-10-17'},
    {'event':'SA2_exam',       'start':'2026-09-28','end':'2026-10-16'},
    {'event':'PSLE_exam',      'start':'2020-08-31','end':'2020-09-25'},
    {'event':'PSLE_exam',      'start':'2021-08-30','end':'2021-09-24'},
    {'event':'PSLE_exam',      'start':'2022-09-01','end':'2022-09-29'},
    {'event':'PSLE_exam',      'start':'2023-08-31','end':'2023-09-28'},
    {'event':'PSLE_exam',      'start':'2024-08-27','end':'2024-09-27'},
    {'event':'PSLE_exam',      'start':'2025-08-25','end':'2025-09-26'},
    {'event':'PSLE_exam',      'start':'2026-08-12','end':'2026-09-30'},
    {'event':'PSLE_results',   'start':'2020-11-25','end':'2020-11-25'},
    {'event':'PSLE_results',   'start':'2021-11-24','end':'2021-11-24'},
    {'event':'PSLE_results',   'start':'2022-11-23','end':'2022-11-23'},
    {'event':'PSLE_results',   'start':'2023-11-22','end':'2023-11-22'},
    {'event':'PSLE_results',   'start':'2024-11-27','end':'2024-11-27'},
    {'event':'PSLE_results',   'start':'2025-11-26','end':'2025-11-26'},
    {'event':'PSLE_results',   'start':'2026-11-24','end':'2026-11-25'},
    {'event':'OLevel_exam',    'start':'2020-10-05','end':'2020-11-06'},
    {'event':'OLevel_exam',    'start':'2021-10-04','end':'2021-11-05'},
    {'event':'OLevel_exam',    'start':'2022-10-03','end':'2022-11-04'},
    {'event':'OLevel_exam',    'start':'2023-10-02','end':'2023-11-03'},
    {'event':'OLevel_exam',    'start':'2024-10-07','end':'2024-11-08'},
    {'event':'OLevel_exam',    'start':'2025-10-06','end':'2025-11-07'},
    {'event':'OLevel_exam',    'start':'2026-10-05','end':'2026-11-06'},
    {'event':'OLevel_results', 'start':'2021-01-11','end':'2021-01-11'},
    {'event':'OLevel_results', 'start':'2022-01-12','end':'2022-01-12'},
    {'event':'OLevel_results', 'start':'2023-01-11','end':'2023-01-11'},
    {'event':'OLevel_results', 'start':'2024-01-10','end':'2024-01-10'},
    {'event':'OLevel_results', 'start':'2025-01-14','end':'2025-01-14'},
    {'event':'OLevel_results', 'start':'2026-01-13','end':'2026-01-15'},
    {'event':'ALevel_exam',    'start':'2020-10-08','end':'2020-11-20'},
    {'event':'ALevel_exam',    'start':'2021-10-07','end':'2021-11-19'},
    {'event':'ALevel_exam',    'start':'2022-10-06','end':'2022-11-18'},
    {'event':'ALevel_exam',    'start':'2023-10-05','end':'2023-11-17'},
    {'event':'ALevel_exam',    'start':'2024-10-10','end':'2024-11-22'},
    {'event':'ALevel_exam',    'start':'2025-10-09','end':'2025-11-21'},
    {'event':'ALevel_exam',    'start':'2026-10-08','end':'2026-11-27'},
    {'event':'ALevel_results', 'start':'2021-02-26','end':'2021-02-26'},
    {'event':'ALevel_results', 'start':'2022-02-25','end':'2022-02-25'},
    {'event':'ALevel_results', 'start':'2023-02-24','end':'2023-02-24'},
    {'event':'ALevel_results', 'start':'2024-02-22','end':'2024-02-22'},
    {'event':'ALevel_results', 'start':'2025-02-21','end':'2025-02-21'},
    {'event':'ALevel_results', 'start':'2026-02-19','end':'2026-02-23'},
]

df_events = pd.DataFrame(SEAB_EVENTS)
df_events['start'] = pd.to_datetime(df_events['start'])
df_events['end']   = pd.to_datetime(df_events['end'])
print(f'{len(SEGMENTS)} segments defined')
print(f'{len(df_events)} exam/results events loaded')

## Section 4: Feature Engineering

Add calendar and exam-urgency features to every row.

Urgency features are level-specific:
- `primary_urgency` — days to PSLE
- `secondary_urgency` — days to O-Level
- `jc_urgency` — days to A-Level
- `sa_urgency` — days to SA1/SA2 (all levels)
- `results_urgency` — days since most recent results release (all levels)

In [ ]:
# Read from Silver Delta table — Gold does not depend on pandas memory
df_silver = spark.read.table(f'{CATALOG}.{SCHEMA}.silver_demand').toPandas()
df_silver['date']   = pd.to_datetime(df_silver['date'])
df_silver['demand'] = pd.to_numeric(df_silver['demand'], errors='coerce')

def build_urgency_series(dates, event_keyword, decay_days=14, direction='future'):
    import re
    filtered = [e for e in SEAB_EVENTS if re.search(event_keyword, e['event'])]
    starts   = [pd.to_datetime(e['start']) for e in filtered]
    scores   = []
    for d in dates:
        d = pd.Timestamp(d)
        if direction == 'future':
            upcoming = [s for s in starts if s >= d]
            delta = (min(upcoming) - d).days if upcoming else 365
        else:
            past = [s for s in starts if s <= d]
            delta = (d - max(past)).days if past else 365
        scores.append(float(np.exp(-delta / decay_days)))
    return scores

def is_school_holiday(d):
    cal = MOE_CALENDAR.get(d.year, {})
    for period in ['march_hols','june_hols','sept_hols','year_end']:
        if period in cal:
            s, e = pd.to_datetime(cal[period][0]), pd.to_datetime(cal[period][1])
            if s <= d <= e:
                return 1
    return 0

# Compute urgency features on unique dates (faster than row-by-row on full table)
unique_dates = df_silver['date'].drop_duplicates().sort_values().reset_index(drop=True)

urgency_df = pd.DataFrame({'date': unique_dates})
urgency_df['primary_urgency']   = build_urgency_series(unique_dates, 'PSLE_exam')
urgency_df['secondary_urgency'] = build_urgency_series(unique_dates, 'OLevel_exam')
urgency_df['jc_urgency']        = build_urgency_series(unique_dates, 'ALevel_exam')
urgency_df['sa_urgency']        = build_urgency_series(unique_dates, 'SA[12]_exam')
urgency_df['results_urgency']   = build_urgency_series(unique_dates, 'results', direction='past')
urgency_df['is_school_holiday'] = unique_dates.apply(is_school_holiday).values

df_features = df_silver.merge(urgency_df, on='date', how='left')

print(f'Read from {CATALOG}.{SCHEMA}.silver_demand')
print(f'Feature table: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns')
display(spark.createDataFrame(df_features.head(10).astype(str)))

### Gold Layer — Feature-Engineered, Model-Ready

Enrich Silver data with all features. Gold is the table the model consumes.
Writing to Delta here means the model can always be retrained from Gold without re-running ingestion.

In [ ]:
# Write feature table to Gold Delta
(spark.createDataFrame(df_features)
     .write.format('delta')
     .mode('overwrite')
     .option('overwriteSchema', 'true')
     .saveAsTable(f'{CATALOG}.{SCHEMA}.gold_features'))

# Read back from Gold — model always consumes from this table
df_features = spark.read.table(f'{CATALOG}.{SCHEMA}.gold_features').toPandas()
df_features['date'] = pd.to_datetime(df_features['date'])

print(f'Gold table written and read back: {CATALOG}.{SCHEMA}.gold_features ({len(df_features):,} rows)')
display(spark.read.table(f'{CATALOG}.{SCHEMA}.gold_features').limit(5))

### Gold Delta Constraints

Table-level rules on `gold_features` ensuring feature values are within expected ranges.
Urgency scores are exponential decay values bounded between 0 and 1.

In [ ]:
# NOT NULL constraints
spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.gold_features ALTER COLUMN date SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.gold_features ALTER COLUMN segment SET NOT NULL")
spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.gold_features ALTER COLUMN demand SET NOT NULL")

# CHECK constraints — urgency scores are exp decay values bounded 0-1
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA}.gold_features
    ADD CONSTRAINT valid_demand_range
    CHECK (demand >= 0 AND demand <= 100)
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA}.gold_features
    ADD CONSTRAINT valid_urgency_scores
    CHECK (
        primary_urgency   >= 0 AND primary_urgency   <= 1 AND
        secondary_urgency >= 0 AND secondary_urgency <= 1 AND
        jc_urgency        >= 0 AND jc_urgency        <= 1 AND
        sa_urgency        >= 0 AND sa_urgency        <= 1 AND
        results_urgency   >= 0 AND results_urgency   <= 1
    )
""")

# Verify
constraints = spark.sql(f"SHOW TBLPROPERTIES {CATALOG}.{SCHEMA}.gold_features") \
                   .filter("key LIKE 'delta.constraints%'")
print(f"Constraints on {CATALOG}.{SCHEMA}.gold_features:")
constraints.show(truncate=False)

## Section 5: Exploratory Data Analysis

In [ ]:
# Monthly demand trend lines per level
monthly = (
    df_features
    .assign(month=lambda x: x['date'].dt.month)
    .groupby(['level','segment','month'])['demand']
    .mean()
    .reset_index()
)
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, lvl in zip(axes, ['Primary','Secondary','JC']):
    for seg in [s for s in SEG_ORDER if s.startswith(lvl)]:
        sub = monthly[(monthly['level'] == lvl) & (monthly['segment'] == seg)]
        ax.plot(sub['month'], sub['demand'], marker='o', markersize=4,
                label=seg.replace(lvl+' ',''))
    ax.set_title(f'{lvl}')
    ax.set_xticks(range(1,13))
    ax.set_xticklabels(month_labels, rotation=45, fontsize=8)
    ax.set_ylabel('Avg Search Volume')
    ax.legend(fontsize=8)

plt.suptitle('Average Monthly Demand by Segment', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: all segments x month
pivot = (
    df_features
    .assign(month=lambda x: x['date'].dt.month)
    .groupby(['segment','month'])['demand']
    .mean()
    .unstack()
)
pivot.columns = month_labels
pivot = pivot.reindex(SEG_ORDER)

plt.figure(figsize=(16, 6))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5)
plt.title('Average Monthly Search Demand — All Segments')
plt.ylabel('')
plt.tight_layout()
plt.show()